# LLM for Security Operations

## What This Is
LLMs are transforming security operations by automating analyst tasks:

- **Alert triage**: Classify alerts by severity, likely root cause, recommended action
- **IOC extraction**: Extract IP addresses, domains, hashes, CVEs from threat reports
- **Report generation**: Convert raw telemetry or investigation notes into structured reports
- **Threat intel synthesis**: Summarize large threat intelligence reports, extract TTPs
- **Playbook execution**: Step-by-step incident response guidance

## Limitations
- **Hallucination**: LLMs may invent CVE details, incorrectly attribute TTPs, or fabricate IOCs
- **Context limits**: Long log files may exceed context windows
- **Latency**: Real-time alert triage requires sub-second decisions; LLM inference is too slow for that tier
- **Confidentiality**: Sending security logs to third-party APIs may violate compliance requirements

In [ ]:
import json
import re
from typing import Dict, List, Tuple, Optional

# IOC extraction pipeline
# In production: combine regex patterns with LLM extraction for context

class IOCExtractor:
    """Extract Indicators of Compromise from text."""
    
    # Regex patterns
    IPV4 = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
    DOMAIN = re.compile(r'\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+'
                        r'(?:com|net|org|io|xyz|ru|cn|info|biz)\b', re.I)
    MD5 = re.compile(r'\b[a-fA-F0-9]{32}\b')
    SHA256 = re.compile(r'\b[a-fA-F0-9]{64}\b')
    CVE = re.compile(r'CVE-\d{4}-\d{4,7}', re.I)
    URL = re.compile(r'https?://[^\s<>"]+', re.I)
    
    def extract(self, text: str) -> Dict[str, List[str]]:
        return {
            'ipv4': list(set(self.IPV4.findall(text))),
            'domains': list(set(self.DOMAIN.findall(text))),
            'md5_hashes': list(set(self.MD5.findall(text))),
            'sha256_hashes': list(set(self.SHA256.findall(text))),
            'cves': list(set(self.CVE.findall(text))),
            'urls': list(set(self.URL.findall(text))),
        }
    
    def filter_false_positives(self, iocs: Dict) -> Dict:
        """Remove common false positives."""
        # Filter private/loopback IPs
        def is_public_ip(ip):
            parts = [int(x) for x in ip.split('.')]
            if parts[0] in [10, 127]: return False
            if parts[0] == 172 and 16 <= parts[1] <= 31: return False
            if parts[0] == 192 and parts[1] == 168: return False
            return True
        
        filtered = dict(iocs)
        filtered['ipv4'] = [ip for ip in iocs['ipv4'] if is_public_ip(ip)]
        return filtered

# Test on a synthetic threat report
threat_report = """
THREAT INTELLIGENCE REPORT - APT29 Campaign

IOCs observed in this campaign:
- C2 servers: 185.220.101.45, 91.108.4.200, 178.62.54.32
- Malicious domains: evil-corp.xyz, download-update.net, cdn-service.io
- Payload hash (MD5): a1b2c3d4e5f67890a1b2c3d4e5f67890
- Dropper hash (SHA256): abc123def456789012345678901234567890123456789012345678901234abcd
- Exploited vulnerability: CVE-2023-23397, CVE-2024-21762
- C2 URL: https://evil-corp.xyz/update/payload?id=12345

Internal IPs (not IOCs): 192.168.1.50, 10.0.0.5, 172.16.0.1
"""

extractor = IOCExtractor()
raw_iocs = extractor.extract(threat_report)
clean_iocs = extractor.filter_false_positives(raw_iocs)

print('IOC Extraction Results:')
for ioc_type, values in clean_iocs.items():
    if values:
        print(f'  {ioc_type}: {values}')


In [ ]:
# Alert triage pipeline using rule-based + LLM-style classification

class AlertTriageSystem:
    """Alert triage using feature extraction + classification."""
    
    SEVERITY_RULES = {
        'critical': [
            lambda a: a.get('in_kev', False),
            lambda a: a.get('is_ransomware_indicator', False),
            lambda a: a.get('data_exfil_mb', 0) > 1000,
            lambda a: a.get('lateral_movement', False) and a.get('admin_access', False),
        ],
        'high': [
            lambda a: a.get('privileged_account', False) and a.get('after_hours', False),
            lambda a: a.get('new_malware_family', False),
            lambda a: a.get('cvss_score', 0) >= 9 and a.get('internet_facing', False),
        ],
        'medium': [
            lambda a: a.get('failed_auth_count', 0) > 10,
            lambda a: a.get('unusual_port', False),
            lambda a: a.get('dns_tunneling', False),
        ],
    }
    
    def triage(self, alert: Dict) -> Dict:
        severity = 'low'
        reasons = []
        
        for sev_level, rules in self.SEVERITY_RULES.items():
            for rule in rules:
                try:
                    if rule(alert):
                        if sev_level == 'critical':
                            severity = 'critical'
                        elif sev_level == 'high' and severity not in ['critical']:
                            severity = 'high'
                        elif sev_level == 'medium' and severity not in ['critical', 'high']:
                            severity = 'medium'
                        reasons.append(f'{sev_level}: {rule.__doc__ or rule}')
                except: pass
        
        return {'severity': severity, 'reasons': reasons, 'alert': alert}

triage = AlertTriageSystem()

alerts = [
    {'in_kev': True, 'cvss_score': 9.8, 'internet_facing': True, 'asset': 'prod-db-01'},
    {'failed_auth_count': 50, 'privileged_account': False, 'asset': 'workstation-22'},
    {'is_ransomware_indicator': True, 'data_exfil_mb': 2000, 'asset': 'fileserver-01'},
    {'unusual_port': True, 'new_malware_family': False, 'asset': 'office-pc-07'},
]

print('Alert Triage Results:')
print('=' * 60)
for alert_result in [triage.triage(a) for a in alerts]:
    asset = alert_result['alert'].get('asset', 'unknown')
    sev = alert_result['severity'].upper()
    print(f'  [{sev:8s}] {asset}')
